# Local RAG Pipeline with HuggingFace Embeddings (Python 3.12)
This notebook demonstrates how to:
- Read local `.txt` and `.pdf` files
    - This is good enough for 'playing' but production would need  many more examples of files AND longer files.
    - A single service would not be enough

In [ ]:
%pip install --upgrade pip

%pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
from config import settings

In [ ]:
docs_path: Path = settings.DATA_DIR
processed_path: Path = settings.PROCESSED_FILE

print(settings)

## Base Strategy Interface

In [ ]:
from pathlib import Path
from abc import ABC, abstractmethod


class FileHandler(ABC):
    @abstractmethod
    def can_handle(self, file_path: Path) -> bool: ## How robust is this?
        pass

    @abstractmethod
    def extract_text(self, file_path: Path) -> str: ## How robust is this? How do we know the text is extracted correctly?
        pass

## The DocumentLoader

In [ ]:
class DocumentLoader:
    def __init__(self):
        self.handlers: list[FileHandler] = []

    def register_handler(self, handler: FileHandler):
        self.handlers.append(handler)

    def extract_text(self, file_path: Path) -> str:
        for handler in self.handlers:
            if handler.can_handle(file_path): ## check if the handler can handle the file type
                ## if yes, call the extract_text method of the handler
                return handler.extract_text(file_path)
        raise ValueError(f"No handler for file type: {file_path.suffix}")

## Text Handler

In [ ]:
class TxtHandler(FileHandler):
    def can_handle(self, file_path: Path) -> bool:
        return file_path.suffix.lower() == ".txt"

    def extract_text(self, file_path: Path) -> str:
        return file_path.read_text(encoding="utf-8")

## PDF Handler
Note that PDFs can be encoded with text or as 'images' of text. This class checks and handles both cases

In [ ]:
import PyPDF2
from pdf2image import convert_from_path
import pytesseract


class PdfHandler(FileHandler):
    def can_handle(self, file_path: Path) -> bool:
        return file_path.suffix.lower() == ".pdf"

    def is_text_based(self, file_path: Path) -> bool:
        try:
            with open(file_path, "rb") as f:
                reader = PyPDF2.PdfReader(f)
                return any(page.extract_text() for page in reader.pages)
        except:
            return False

    def extract_text(self, file_path: Path) -> str:
        if self.is_text_based(file_path):
            with open(file_path, "rb") as f:
                reader = PyPDF2.PdfReader(f)
                return "\n".join(page.extract_text() or "" for page in reader.pages)
        else:
            pages = convert_from_path(file_path)
            return "\n".join(pytesseract.image_to_string(p) for p in pages)

In [ ]:
import re

def clean_text(text: str) -> str:
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'Page \d+', '', text, flags=re.IGNORECASE)
    text = ''.join(c for c in text if c.isprintable())
    return text.strip()

In [ ]:
loader = DocumentLoader()
loader.register_handler(TxtHandler())
loader.register_handler(PdfHandler())

In [ ]:
from dataclasses import dataclass
from typing import List, Dict

@dataclass
class TextFile:
    filename: str
    content: str


all_texts: List[TextFile] = []

In [ ]:
for file in docs_path.rglob("*"): ## Is this recursive? Yes, rglob is recursive
    try:
        content = loader.extract_text(file)
        cleaned = clean_text(content)
        all_texts.append(TextFile(filename=file.name, content=cleaned))
    except Exception as e:
        print(f"Error with {file.name}: {e}")

## Check the documents have been processed

In [ ]:
print(f"Extracted {len(all_texts)} documents.")

In [ ]:
print(all_texts[1])

In [ ]:
from datetime import date

with open(processed_path, "w", encoding="utf-8") as f:  ## Open the file in write mode
    for doc in all_texts: ## Loop through the documents
        ## Maybe add the meatadata as JSON?
        f.write(f"__META__FILE__NAME: {doc.filename}\n")
        f.write(f"__CONTENT__: {doc.content}\n")
        f.write(f"__META__DATE: {date.today().isoformat()}\n")
        f.write(f"__META__SOURCE: internal_docs")

        if settings.DEBUG:
            f.write("\n" + "="*80 + "\n\n") ## Document separator
        else:
            f.write("\n")